In [ ]:
import boto3
import csv
import json
import time
import random

# ---- Config ----
STREAM_NAME = "ecomm-events-stream"
CSV_FILE = "2026-Jun-sample.csv"
REGION = "us-east-1"
BATCH_SIZE = 20
DELAY_SECONDS = 2

# ---- Boto3 client ----
client = boto3.client("kinesis", region_name=REGION)

def send_batch(records):
    """Records ko Kinesis Data Stream ke required format mein bhejta hai"""
    entries = []
    for record in records:
        entries.append({
            "Data": json.dumps(record).encode("utf-8"),
            "PartitionKey": str(record.get("user_id", "default"))
        })

    response = client.put_records(
        StreamName=STREAM_NAME,
        Records=entries
    )
    failed = response.get("FailedRecordCount", 0)
    if failed > 0:
        print(f"⚠️  {failed} records fail hue")
    else:
        print(f"✅ {len(entries)} records stream mein bhej diye")

def main():
    with open(CSV_FILE, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        batch = []
        total_sent = 0

        for row in reader:
            batch.append(row)
            if len(batch) >= BATCH_SIZE:
                send_batch(batch)
                total_sent += len(batch)
                batch = []
                time.sleep(DELAY_SECONDS + random.uniform(-0.5, 0.5))

        if batch:
            send_batch(batch)
            total_sent += len(batch)

        print(f"\n🎉 Total {total_sent} records stream ko bhej diye gaye")

if __name__ == "__main__":
    main()